# Data cleaning and transformation
Working file: `data/trade_raw_2019_2023.csv`, an illustrative extract with realistic quality problems.

In [ ]:
import pandas as pd

raw = pd.read_csv(
    "../../data/trade_raw_2019_2023.csv")
print(raw.shape)
print(raw.dtypes)
raw.sample(5, random_state=1)

## Standardise column names and text

In [ ]:
df = raw.copy()
df.columns = df.columns.str.lower()
df["reporter"] = df["reporter"].str.strip()
print(df["reporter"].value_counts().head(8))

In [ ]:
NAME_MAP = {
    "USA": "United States",
    "U.S.": "United States",
    "united states": "United States",
    "UK": "United Kingdom",
    "U.K.": "United Kingdom",
    "Vietnam": "Viet Nam",
    "germany": "Germany",
    "Federal Republic of Germany": "Germany",
}
rep = df["reporter"].replace(NAME_MAP)
df["reporter"] = rep
print(df["reporter"].nunique(), "reporters")

## Fix numeric values

In [ ]:
value = (df["value"].astype(str)
         .str.replace(",", "", regex=False))
df["value"] = pd.to_numeric(value,
                            errors="coerce")
print(df["value"].isna().sum(), "missing")
print((df["value"] < 0).sum(), "negative")

## Handle missing values

In [ ]:
print(df.isna().sum())

# Options: drop, fill, or flag
df["value_missing"] = df["value"].isna()
dropped = df.dropna(subset=["value"])
print(len(df), "->", len(dropped))

## Manage duplicates

In [ ]:
print(df.duplicated().sum(), "exact dups")
df = df.drop_duplicates()

key = ["reporter", "partner", "period"]
n = df.duplicated(subset=key).sum()
print(n, "rows share a key")

## Standardise units

In [ ]:
thousands = df["unit"] == "USD thousands"
df.loc[thousands, "value"] /= 1000
df.loc[thousands, "unit"] = "USD millions"
print(df["unit"].value_counts())

## Dates and times

In [ ]:
FORMATS = ["%Y-%m-%d", "%d/%m/%Y",
           "%b %Y", "%Y"]

def parse_period(p):
    for f in FORMATS:
        try:
            return pd.to_datetime(p, format=f)
        except ValueError:
            continue
    return pd.NaT

df["date"] = df["period"].map(parse_period)
df["year"] = df["date"].dt.year
df[["period", "date", "year"]].head()

## Combine datasets

In [ ]:
countries = pd.read_excel(
    "../../data/countries.xlsx")
clean = df.merge(
    countries[["iso3", "country_name"]],
    left_on="reporter",
    right_on="country_name",
    how="left", validate="many_to_one",
    indicator=True)
print(clean["_merge"].value_counts())

In [ ]:
wb = pd.read_csv(
    "../../data/wb_indicators_2015_2023.csv")
world = clean[clean["partner"] == "World"]
combined = world.merge(
    wb[["iso3", "year", "gdp_usd"]],
    on=["iso3", "year"], how="left")
combined["exports_pct_gdp"] = (
    combined["value"] * 1e6
    / combined["gdp_usd"] * 100).round(1)
combined[["reporter", "year",
          "exports_pct_gdp"]].head()

## Validate

In [ ]:
def validate(df):
    problems = []
    if df["value"].lt(0).any():
        problems.append("negative values")
    if df["iso3"].isna().any():
        problems.append("unmatched reporters")
    key = ["iso3", "partner", "year"]
    if df.duplicated(key).any():
        problems.append("duplicate keys")
    return problems

print(validate(clean) or "All checks passed")